<a href="https://colab.research.google.com/github/Alejandro7337/puzzle-8/blob/main/Puzzle_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import copy
import heapq
import random
from collections import deque

In [ ]:
class Estadopuzzle:
  #nada que decir definimos lo indefinido xd
  def __init__(self, matriz, padre=None, movimiento=None, profundidad=0):
    self.matriz = matriz
    self.padre = padre
    self.movimiento = movimiento
    self.profundidad = profundidad
    #esto es una clase de contrusctor de java pero en la pyton :V, para guardar el estado anterior y reconstruirlo, junto con los movimientos. dirian que son reglas pero no toy seguro

  def __eq__(self, other):
    return self.matriz == other.matriz
    #aqui comparamos los estados con el eq de equitacion xD

  def __hash__(self):
    return hash(str(self.matriz))
    #lo metemos dentro de un hash para usarlo como conjuntos, por que no se. pero asi me o explicaron xD


In [ ]:
def encontrar_cero(matriz):

    #miramos la matriz, y vamos buscando en la matriz como es 3 por 3 pos aja 3 columnas 3 filas, ademas esto lo vimos en analisis
    for i in range(3):
      for j in range(3):
          #aqui pos cuando lo encontramos devolvemos la posi
          if matriz[i][j] == 0:
            return  i,j

def generar_sucesores(estado):

    movimientos = []

    x, y = encontrar_cero(estado.matriz)

    #no se si neceita explicacion de esto pero me pregunta xD
    direcciones = {
        "abajo": (1,0),
        "izquierda": (0,-1),
        "derecha": (0,1),
        "arriba": (-1, 0)
    }
    #ahora la estresante, provamos todito
    for mov,(dx,dy) in direcciones.items():

        nx, ny = x + dx, y + dy

        if 0 <= nx < 3 and 0 <= ny < 3:

            #copiamos la matriz, un vakup

            nueva_matriz = copy.deepcopy(estado.matriz)

            #intercambiamos
            nueva_matriz[x][y], nueva_matriz[nx][ny]= nueva_matriz[nx][ny], nueva_matriz[x][y]

            nuevo_estado = Estadopuzzle(
                nueva_matriz,
                estado,     #explicar esto despues !no olvidar¡
                mov,
                estado.profundidad + 1
                )

            movimientos.append(nuevo_estado)
    return movimientos

#METODOS DE BUSQUEDA

###Primero en amplitud (dfs)

In [ ]:
#algoritmo bfs amplitud
def bfs(estado_inicial_matriz, estado_objetivo):

    estado_inicial_obj = Estadopuzzle(estado_inicial_matriz)
    cola = deque([estado_inicial_obj])

    visitados = set()
    visitados.add(str(estado_inicial_obj.matriz)) # Add initial state's matrix to visited

    nodos_generados = 0
    nodos_visitados = 0

    while cola:
        estado_actual = cola.popleft()

        if estado_actual.matriz == estado_objetivo:
            print("Lo logro señor")
            print("lo logre :D")
            print("✅ Solución encontrada con BFS")
            print("nodos generados", nodos_generados)
            print("nodos visitados", nodos_visitados) #no esta funcionando bien pero no es necesario. creo
            return estado_actual

        #sucesores
        for sucesor in generar_sucesores(estado_actual):

          nodos_generados += 1

          if str(sucesor.matriz) not in visitados:
            visitados.add(str(sucesor.matriz))
            cola.append(sucesor)
            nodos_visitados += 1 # Count nodes added to queue as visited
    return None

###Primero en profundidad

In [ ]:
def dfs(estado_inicial, estado_objetivo):

    # En DFS usamos una PILA, pa andar ready for plyer one xD
    pila = []

    # mas fazil
    visitados = set()

    #contamos nodos .D
    nodos_generados = 0
    nodos_visitados = 0

    # Creamos el estado inicial, me aburr de escribir ya mira que hace el resto :V
    estado_inicio = Estadopuzzle(estado_inicial)

    pila.append(estado_inicio)

    while pila:

        estado_actual = pila.pop()

        if str(estado_actual.matriz) in visitados:
            continue

        visitados.add(str(estado_actual.matriz))
        nodos_visitados += 1

        if estado_actual.matriz == estado_objetivo:
            print("✅ Solución encontrada con DFS")
            print("Nodos generados:", nodos_generados)
            print("Nodos visitados:", nodos_visitados)
            return estado_actual

        sucesores = generar_sucesores(estado_actual)

        for sucesor in generar_sucesores(estado_actual):
            nodos_generados += 1
            if str(sucesor.matriz) not in visitados:
                pila.append(sucesor)

    return None

###A*

In [ ]:
def distancia_manhattan(estado, objetivo):
    distancia_total = 0
    for i in range(3):
        for j in range(3):
            valor = estado[i][j]
            if valor != 0:
                for x in range(3):
                    for y in range(3):
                        if objetivo[x][y] == valor:
                            distancia_total += abs(i - x) + abs(j - y)
    return distancia_total

def a(estado_inicial, estado_objetivo):
    frontera = []
    # CORRECCIÓN: usar str() de forma consistente en todo el método
    visitados = set()
    nodos_generados = 0
    nodos_visitados = 0
    contador_push = 0

    estado_inicio = Estadopuzzle(estado_inicial)
    g = 0
    h = distancia_manhattan(estado_inicial, estado_objetivo)
    f = g + h
    heapq.heappush(frontera, (f, contador_push, estado_inicio))
    contador_push += 1

    while frontera:
        _, _, estado_actual = heapq.heappop(frontera)

        # CORRECCIÓN: verificar con str() igual que se agrega
        if str(estado_actual.matriz) in visitados:
            continue

        # CORRECCIÓN: agregar con str() (antes usaba tuple, inconsistente)
        visitados.add(str(estado_actual.matriz))
        nodos_visitados += 1

        if estado_actual.matriz == estado_objetivo:
            print("✅ Solución encontrada con A*")
            print("Nodos generados:", nodos_generados)
            print("Nodos visitados:", nodos_visitados)
            return estado_actual

        for sucesor in generar_sucesores(estado_actual):
            nodos_generados += 1
            if str(sucesor.matriz) not in visitados:
                g = sucesor.profundidad
                h = distancia_manhattan(sucesor.matriz, estado_objetivo)
                f = g + h
                heapq.heappush(frontera, (f, contador_push, sucesor))
                contador_push += 1
    return None


###Ascenso en Colina

In [ ]:
def asc(estado_inicial, estado_objetivo, max_reinicios=20):

    def intentar_desde(estado_partida):
        ABIERTO = []
        CERRADO = []
        nodoInicial = Estadopuzzle(estado_partida)
        nodoMeta    = estado_objetivo
        nodos_generados = 0
        ABIERTO.append(nodoInicial)

        while ABIERTO:
            nodo = ABIERTO.pop(0)
            if str(nodo.matriz) not in [str(c.matriz) for c in CERRADO]:
                CERRADO.append(nodo)
                if nodo.matriz == nodoMeta:
                    return nodo, nodos_generados, len(CERRADO)
                sucesores = generar_sucesores(nodo)
                nodos_generados += len(sucesores)
                sucesores_con_h = [
                    (distancia_manhattan(s.matriz, nodoMeta), s)
                    for s in sucesores
                    if str(s.matriz) not in [str(c.matriz) for c in CERRADO]
                ]
                if sucesores_con_h:
                    es_meta = [(h, s) for h, s in sucesores_con_h if s.matriz == nodoMeta]
                    if es_meta:
                        ABIERTO = [s for _, s in es_meta] + ABIERTO
                        continue
                    h_actual = distancia_manhattan(nodo.matriz, nodoMeta)
                    mejor_h = min(h for h, _ in sucesores_con_h)
                    if mejor_h < h_actual:
                        sucesores_con_h.sort(key=lambda x: x[0])
                        ABIERTO = [s for _, s in sucesores_con_h] + ABIERTO
                    else:
                        return None, nodos_generados, len(CERRADO)

        return None, nodos_generados, len(CERRADO)

    nodos_totales   = 0
    visitados_total = 0

    for reinicio in range(max_reinicios + 1):
        if reinicio == 0:
            estado_partida = estado_inicial
        else:
            plano = [n for fila in estado_inicial for n in fila]
            random.shuffle(plano)
            estado_partida = [plano[i:i+3] for i in range(0, 9, 3)]
            print(f"🔄 Reinicio aleatorio #{reinicio}")

        resultado, ng, nv = intentar_desde(estado_partida)
        nodos_totales   += ng
        visitados_total += nv

        if resultado is not None:
            print("✅ Solución encontrada con Ascenso en Colina")
            print(f"Reinicios realizados: {reinicio}")
            print(f"Nodos generados: {nodos_totales}")
            print(f"Nodos visitados: {visitados_total}")
            return resultado

    print("❌ Fracaso: Ascenso en Colina no encontró solución tras todos los reinicios")
    print(f"Nodos generados: {nodos_totales}")
    print(f"Nodos visitados: {visitados_total}")
    return None

###Algoritmo Genético

In [ ]:
def gen(estado_inicial, estado_objetivo,
        tam_poblacion=100, tasa_mutacion=0.1, longitud_cromosoma=50):

    CADA_M_GENERACIONES = 5
    MAX_GENERACIONES    = 500
    MOVS = ["arriba", "abajo", "izquierda", "derecha"]
    DELTAS = {"arriba":(-1,0), "abajo":(1,0), "izquierda":(0,-1), "derecha":(0,1)}

    # ── Optimización 1: representar estado como tupla plana + pos del cero ──
    # Evita deepcopy en cada evaluación de fitness (9x más rápido)

    def matriz_a_tupla(matriz):
        return tuple(n for fila in matriz for n in fila)

    def tupla_a_matriz(t):
        return [list(t[i:i+3]) for i in range(0, 9, 3)]

    def encontrar_cero_tupla(t):
        idx = t.index(0)
        return idx // 3, idx % 3

    # ── Optimización 2: aplicar movimientos sobre tupla sin deepcopy ────────

    def aplicar_secuencia(estado_t, cero_pos, secuencia):
        """
        Aplica movimientos sobre tupla plana.
        Retorna (estado_final_tupla, historial_de_tuplas, movimientos_validos).
        Movimientos inválidos se omiten del historial pero se penalizan en fitness.
        """
        estado = list(estado_t)
        ci, cj  = cero_pos
        historial      = [tuple(estado)]
        movs_validos   = []
        penalizacion   = 0

        for mov in secuencia:
            dx, dy = DELTAS[mov]
            ni, nj = ci + dx, cj + dy

            if not (0 <= ni < 3 and 0 <= nj < 3):
                penalizacion += 2   # movimiento inválido
                continue

            idx1 = ci * 3 + cj
            idx2 = ni * 3 + nj
            estado[idx1], estado[idx2] = estado[idx2], estado[idx1]
            ci, cj = ni, nj
            historial.append(tuple(estado))
            movs_validos.append(mov)

        return tuple(estado), (ci, cj), historial, movs_validos, penalizacion

    # ── Optimización 3: caché de fitness con diccionario ────────────────────
    # Evita recalcular el mismo cromosoma dos veces

    cache_fitness = {}

    def fitness(cromosoma):
        key = tuple(cromosoma)
        if key in cache_fitness:
            return cache_fitness[key]

        estado_final, _, _, _, penalizacion = aplicar_secuencia(
            estado_inicial_t, cero_inicial, cromosoma
        )
        resultado = distancia_manhattan(
            tupla_a_matriz(estado_final), estado_objetivo
        ) + penalizacion
        cache_fitness[key] = resultado
        return resultado

    # ── Optimización 4: cruce y mutación sobre listas (sin copy innecesario) ─

    def cruce(p1, p2):
        punto = random.randint(1, longitud_cromosoma - 1)
        return p1[:punto] + p2[punto:], p2[:punto] + p1[punto:]

    def mutar(cromosoma):
        c = cromosoma[:]
        c[random.randint(0, longitud_cromosoma - 1)] = random.choice(MOVS)
        return c

    def seleccionar(poblacion, fitnesses):
        candidatos = random.sample(list(zip(fitnesses, poblacion)), 3)
        return min(candidatos, key=lambda x: x[0])[1]

    def individuo_aleatorio():
        return [random.choice(MOVS) for _ in range(longitud_cromosoma)]

    # ── Inicialización ───────────────────────────────────────────────────────

    estado_inicial_t = matriz_a_tupla(estado_inicial)
    objetivo_t       = matriz_a_tupla(estado_objetivo)
    cero_inicial     = encontrar_cero_tupla(estado_inicial_t)

    poblacion       = [individuo_aleatorio() for _ in range(tam_poblacion)]
    fitnesses       = [fitness(c) for c in poblacion]
    nodos_generados = tam_poblacion
    terminado       = False
    solucion        = None
    generacion      = 0

    print(f"Generación 0 | Mejor fitness: {min(fitnesses)} | Peor: {max(fitnesses)}")

    # ── Mientras no terminado ────────────────────────────────────────────────

    while not terminado and generacion < MAX_GENERACIONES:

        generacion += 1

        # Producir nueva generación con n individuos
        nueva_generacion = []

        # Elitismo: conservar el mejor individuo sin cambios
        mejor_idx = min(range(len(fitnesses)), key=lambda i: fitnesses[i])
        nueva_generacion.append(poblacion[mejor_idx])

        # Para n individuos
        while len(nueva_generacion) < tam_poblacion:

            # Seleccionar parejas de la generación anterior
            padre1 = seleccionar(poblacion, fitnesses)
            padre2 = seleccionar(poblacion, fitnesses)

            # Cruzar obteniendo descendientes
            hijo1, hijo2 = cruce(padre1, padre2)
            nodos_generados += 2

            # Mutar los descendientes cada m generaciones
            if generacion % CADA_M_GENERACIONES == 0:
                if random.random() < tasa_mutacion:
                    hijo1 = mutar(hijo1)
                if random.random() < tasa_mutacion:
                    hijo2 = mutar(hijo2)

            nueva_generacion.append(hijo1)
            if len(nueva_generacion) < tam_poblacion:
                nueva_generacion.append(hijo2)

        # Insertar descendientes en la nueva generación
        poblacion = nueva_generacion
        fitnesses = [fitness(c) for c in poblacion]
        mejor_fitness = min(fitnesses)

        if generacion % 50 == 0:
            print(f"Generación {generacion} | Mejor fitness: {mejor_fitness}")

        # Si la población converge → Terminado = TRUE
        if mejor_fitness == 0:
            terminado = True
            solucion  = poblacion[fitnesses.index(0)]
            print(f"✅ Solución encontrada con Algoritmo Genético")
            print(f"Generación de convergencia: {generacion}")
            print(f"Nodos generados: {nodos_generados}")

    # ── Reportar éxito: construir cadena de Estadopuzzle ────────────────────

    if solucion:
        _, _, historial, movs_validos, _ = aplicar_secuencia(
            estado_inicial_t, cero_inicial, solucion
        )

        # Recortar hasta el primer estado que alcance nodoMeta
        estados_hasta_meta = []
        movs_hasta_meta    = []
        for i, estado_t in enumerate(historial):
            estados_hasta_meta.append(estado_t)
            if estado_t == objetivo_t:
                movs_hasta_meta = movs_validos[:i]
                break

        # Si por alguna razón no alcanzó exactamente, usar el historial completo
        if not any(t == objetivo_t for t in historial):
            estados_hasta_meta = historial
            movs_hasta_meta    = movs_validos

        # Construir cadena de Estadopuzzle enlazados por padre
        nodo_anterior = None
        for i, estado_t in enumerate(estados_hasta_meta):
            mov  = None if i == 0 else movs_hasta_meta[i - 1]
            nodo = Estadopuzzle(
                tupla_a_matriz(estado_t),
                padre=nodo_anterior,
                movimiento=mov,
                profundidad=i
            )
            nodo_anterior = nodo

        return nodo_anterior

    # Si la población no converge → fracaso
    print(f"❌ Fracaso: no convergió en {MAX_GENERACIONES} generaciones")
    print(f"Mejor fitness alcanzado: {min(fitnesses)}")
    print(f"Nodos generados: {nodos_generados}")
    return None

###Reconstruir Camino

In [ ]:

def reconstruir_camino(estado):
    camino = []
    while estado:
        camino.append(estado)
        estado = estado.padre
    camino.reverse()
    for paso in camino:
        print("Movimiento:", paso.movimiento)
        mostrar_puzzle(paso.matriz)


#Revisar si es soluble

In [ ]:
def contar_inversiones(puzzle):
    """Cuenta inversiones ignorando el 0."""
    plano = [n for fila in puzzle for n in fila if n != 0]
    inversiones = 0
    for i in range(len(plano)):
        for j in range(i + 1, len(plano)):
            if plano[i] > plano[j]:
                inversiones += 1
    return inversiones

def es_resoluble(inicial, objetivo):
    """
    El puzzle es resoluble si inicial y objetivo
    tienen la misma paridad de inversiones.
    """
    return contar_inversiones(inicial) % 2 == contar_inversiones(objetivo) % 2

# UTILIDADES DE LECTURA DE ARCHIVO

In [ ]:
def obtener_ruta(file_obj):
    """
    Maneja ambas versiones de Gradio:
      - Gradio 3.x → objeto con atributo .name
      - Gradio 4.x → string con la ruta directamente
    """
    if file_obj is None:
        return None
    if isinstance(file_obj, str):
        return file_obj
    if hasattr(file_obj, 'name'):
        return file_obj.name
    # Gradio 4.x puede retornar un dict
    if isinstance(file_obj, dict) and 'name' in file_obj:
        return file_obj['name']
    return str(file_obj)



In [ ]:
def leer_puzzle_desde_archivo(file_obj):
    """Lee y valida un puzzle desde el objeto archivo que entrega Gradio."""
    ruta = obtener_ruta(file_obj)
    if ruta is None:
        raise ValueError("No se recibió ningún archivo.")

    try:
        with open(ruta, 'r') as f:
            contenido = f.read()
    except Exception as e:
        raise ValueError(f"No se pudo abrir el archivo: {e}")

    puzzle = []
    lineas = [l for l in contenido.strip().splitlines() if l.strip()]

    if len(lineas) != 3:
        raise ValueError(
            f"El archivo debe tener exactamente 3 filas, tiene {len(lineas)}."
        )

    for num_linea, linea in enumerate(lineas, start=1):
        partes = linea.split()
        if len(partes) != 3:
            raise ValueError(
                f"Línea {num_linea}: debe tener 3 números, tiene {len(partes)}."
            )
        fila = []
        for valor in partes:
            try:
                fila.append(int(valor))
            except ValueError:
                raise ValueError(f"'{valor}' no es un número entero válido.")
        puzzle.append(fila)

    numeros = [n for fila in puzzle for n in fila]
    if numeros.count(0) != 1:
        raise ValueError("El puzzle debe contener exactamente un 0 (espacio vacío).")
    if sorted(numeros) != list(range(9)):
        raise ValueError(
            "El puzzle debe contener exactamente los números del 0 al 8."
        )
    return puzzle



In [ ]:
def generar_objetivo(puzzle):
    numeros = [n for fila in puzzle for n in fila]
    orden = sorted(numeros)
    orden.remove(0)
    orden.append(0)
    return [orden[i:i+3] for i in range(0, 9, 3)]

# VISUALIZACIÓN HTML DEL PUZZLE

In [ ]:
def puzzle_a_html(matriz, titulo=""):
    """Convierte una matriz 3x3 en una tabla HTML visual."""
    colores = {
        0: ("#1a1a2e", "#ffffff"),   # vacío: oscuro
        1: ("#e94560", "#ffffff"),
        2: ("#0f3460", "#ffffff"),
        3: ("#533483", "#ffffff"),
        4: ("#05c46b", "#ffffff"),
        5: ("#ffd32a", "#000000"),
        6: ("#ff5e57", "#ffffff"),
        7: ("#0fbcf9", "#000000"),
        8: ("#f8b739", "#000000"),
    }

    html = f"""
    <div style="display:inline-block; margin:6px; text-align:center;">
        <div style="font-size:11px; color:#888; margin-bottom:4px;">{titulo}</div>
        <table style="border-collapse:collapse;">
    """
    for fila in matriz:
        html += "<tr>"
        for val in fila:
            bg, fg = colores.get(val, ("#333", "#fff"))
            texto = str(val) if val != 0 else "·"
            html += f"""
            <td style="
                width:52px; height:52px;
                background:{bg}; color:{fg};
                font-size:22px; font-weight:bold;
                text-align:center; vertical-align:middle;
                border:3px solid #111;
                border-radius:6px;
            ">{texto}</td>
            """
        html += "</tr>"
    html += "</table></div>"
    return html

In [ ]:
def generar_html_camino(camino):
    """Genera el HTML completo con todos los estados del camino."""
    if not camino:
        return "<p style='color:red'>Sin camino.</p>"

    html = "<div style='overflow-x:auto; white-space:nowrap; padding:10px;'>"
    for i, paso in enumerate(camino):
        if i == 0:
            etiqueta = "Inicial"
        elif i == len(camino) - 1:
            etiqueta = "✅ Final"
        else:
            mov = paso.movimiento or ""
            etiqueta = f"↓ {mov.capitalize()}"
        html += puzzle_a_html(paso.matriz, etiqueta)
        if i < len(camino) - 1:
            html += "<span style='font-size:28px; vertical-align:middle; color:#aaa; margin:0 2px'>→</span>"
    html += "</div>"
    return html

# INTERFAZ


In [ ]:
# Para actualizar gradio (a veces molesta)
!pip install --upgrade gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing installation: gradio 5.50.0
    Uninstalling gradio-5.50.0:
      Successfully uninstalled gradio-5.50.0


##Funcion principal de la interfaz

In [ ]:
def process_inputs(archivo_inicial, opcion_objetivo, archivo_objetivo,
                   metodo, tam_poblacion, tasa_mutacion, longitud_cromosoma):

    # --- 1. Leer estado inicial ---
    if archivo_inicial is None:
        return ("<p style='color:red'>❌ Suba el archivo del estado inicial.</p>", "", "")
    try:
        estado_inicial = leer_puzzle_desde_archivo(archivo_inicial)
    except Exception as e:
        return (f"<p style='color:red'>❌ Error en archivo inicial: {e}</p>", "", "")

    # --- 2. Determinar estado objetivo ---
    if opcion_objetivo == "Automático":
        estado_objetivo = generar_objetivo(estado_inicial)
    else:
        if archivo_objetivo is None:
            return ("<p style='color:red'>❌ Suba el archivo del estado objetivo.</p>", "", "")
        try:
            estado_objetivo = leer_puzzle_desde_archivo(archivo_objetivo)
        except Exception as e:
            return (f"<p style='color:red'>❌ Error en archivo objetivo: {e}</p>", "", "")

    # --- 3. Verificar solucionabilidad ---
    if not es_resoluble(estado_inicial, estado_objetivo):
        return (
            "<p style='color:orange; font-size:15px'>⚠️ Esta configuración <b>no tiene solución</b>.<br>"
            "El estado inicial y el objetivo tienen distinta paridad de inversiones.<br>"
            "Son matemáticamente inalcanzables entre sí.</p>",
            "Paridades incompatibles — sin solución posible.",
            ""
        )

    # --- 4. Seleccionar método y ejecutar capturando prints ---
    mapa_metodos = {
        "Primero en Amplitud (BFS)": bfs,
        "Primero en Profundidad (DFS)": dfs,
        "A* (Manhattan)": a,
        "Ascenso en Colina": asc,
        "Algoritmo Genético": gen,
    }
    if metodo not in mapa_metodos:
        return (f"<p style='color:red'>❌ Método no implementado: {metodo}</p>", "", "")

    funcion = mapa_metodos[metodo]
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer
    resultado = None
    try:
        if metodo == "Algoritmo Genético":
          resultado = funcion(estado_inicial, estado_objetivo,
                              tam_poblacion=int(tam_poblacion),
                              tasa_mutacion=float(tasa_mutacion),
                              longitud_cromosoma=int(longitud_cromosoma))
        else:
            resultado = funcion(estado_inicial, estado_objetivo)
    except Exception as e:
        sys.stdout = old_stdout
        return (f"<p style='color:red'>❌ Error durante la búsqueda: {e}</p>", "", "")
    finally:
        sys.stdout = old_stdout

    log_busqueda = buffer.getvalue()

    # --- 5. Sin solución ---
    if resultado is None:
        return (
            "<p style='color:orange'>⚠️ No se encontró solución para esta configuración.</p>",
            log_busqueda,
            ""
        )

    # --- 6. Reconstruir camino ---
    camino = []
    estado = resultado
    while estado:
        camino.append(estado)
        estado = estado.padre
    camino.reverse()

    # --- 7. Movimientos ---
    lineas_movimientos = []
    for i, paso in enumerate(camino):
        if i == 0:
            lineas_movimientos.append("Paso 0 | Estado INICIAL")
        else:
            lineas_movimientos.append(f"Paso {i} | Movimiento: {paso.movimiento.upper()}")
        for fila in paso.matriz:
            lineas_movimientos.append("  " + "  ".join(
                ["[·]" if v == 0 else f"[{v}]" for v in fila]
            ))
        lineas_movimientos.append("")
    texto_movimientos = "\n".join(lineas_movimientos)

    # --- 8. Estadísticas ---
    stats_html = f"""
    <div style="background:#1a1a2e; color:#fff !important; padding:16px;
        border-radius:10px; font-family:monospace; font-size:14px;">
        <b style="font-size:16px; color:#fff">📊 Estadísticas — {metodo}</b><br><br>
        {log_busqueda.replace(chr(10), '<br>')}
        <br>
        <b style="color:#fff">Pasos para resolver:</b> <span style="color:#fff">{len(camino) - 1}</span><br>
        <b style="color:#fff">Profundidad de la solución:</b> <span style="color:#fff">{resultado.profundidad}</span>
    </div>
    """

    # --- 9. HTML del camino ---
    html_camino = generar_html_camino(camino)
    return html_camino, stats_html, texto_movimientos

## INTERFAZ GRADIO

In [ ]:
import gradio as gr
import io
import sys

# ── Fix 1: CSS va en launch(), no en Blocks (Gradio 6.x lo movió) ──────
css = """
.gradio-container { font-family: 'Segoe UI', sans-serif; }
.output-html { overflow-x: auto !important; }
"""

def toggle_objetivo(opcion):
    return gr.update(visible=(opcion == "Manual"))

# ── Fix 2: en lugar de 3 updates simultáneos, usar un solo Group ────────
# Los sliders viven dentro de un gr.Group que se muestra/oculta de una vez
def toggle_params_gen(metodo):
    visible = metodo == "Algoritmo Genético"
    return gr.update(visible=visible)   # ← UN solo update, no tres


with gr.Blocks(title="Puzzle 8") as demo:

    gr.Markdown("""
    # 🧩 Puzzle 8 — Búsqueda Heurística
    Hecho por Alejandro Alba Malagón y David Santiago Mahecha Cruz

    Suba archivos `.txt` con el estado del puzzle.
    Formato: 3 filas × 3 números separados por espacio. Use `0` para el espacio vacío.
```
    1 2 3
    4 5 6
    7 0 8
```
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📂 Archivos de entrada")

            archivo_inicial = gr.File(
                label="📄 Estado inicial (.txt)",
                height=80
            )
            opcion_objetivo = gr.Radio(
                choices=["Automático", "Manual"],
                value="Automático",
                label="Estado objetivo"
            )
            archivo_objetivo = gr.File(
                label="📄 Estado objetivo (.txt)",
                height=80,
                visible=False
            )
            metodo = gr.Dropdown(
                choices=[
                    "Primero en Amplitud (BFS)",
                    "Primero en Profundidad (DFS)",
                    "Ascenso en Colina",
                    "A* (Manhattan)",
                    "Algoritmo Genético"
                ],
                value="Primero en Amplitud (BFS)",
                label="🔍 Método de búsqueda"
            )

            # ── Fix 2: los 3 sliders dentro de un Group ─────────────────
            # Un solo update de visibilidad en lugar de 3 simultáneos
            with gr.Group(visible=False) as grupo_gen:
                tam_poblacion = gr.Slider(
                    minimum=20, maximum=500, value=100, step=10,
                    label="🧬 Tamaño de población"
                )
                longitud_cromosoma = gr.Slider(
                    minimum=20, maximum=100, value=50, step=5,
                    label="🧬 Longitud del cromosoma (movimientos)"
                )
                tasa_mutacion = gr.Slider(
                    minimum=0.01, maximum=0.5, value=0.1, step=0.01,
                    label="🧬 Tasa de mutación"
                )

            boton = gr.Button("▶ Resolver", variant="primary", size="lg")

        with gr.Column(scale=2):
            gr.Markdown("### 📊 Estadísticas y nodos")
            stats_out = gr.HTML(
                value="<p style='color:#888'>Esperando ejecución...</p>"
            )

    gr.Markdown("### 🔄 Secuencia de estados (Inicial → Final)")
    camino_out = gr.HTML(
        value="<p style='color:#888'>Aquí aparecerán los estados del puzzle.</p>"
    )

    gr.Markdown("### 📋 Secuencia de movimientos")
    movimientos_out = gr.Textbox(
        label="Movimientos paso a paso",
        lines=15,
        interactive=False
    )

    # ── Event handlers ───────────────────────────────────────────────────
    opcion_objetivo.change(
        fn=toggle_objetivo,
        inputs=opcion_objetivo,
        outputs=archivo_objetivo
    )
    metodo.change(
        fn=toggle_params_gen,
        inputs=metodo,
        outputs=grupo_gen        # ← un solo componente, no una lista de 3
    )
    boton.click(
        fn=process_inputs,
        inputs=[archivo_inicial, opcion_objetivo, archivo_objetivo,
                metodo, tam_poblacion, tasa_mutacion, longitud_cromosoma],
        outputs=[camino_out, stats_out, movimientos_out]
    )

# ── Fix 3: CSS va aquí en launch(), queue evita bloqueos del servidor ───
demo.launch(debug=True, css=css)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ccac6f88fb32035a84.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ccac6f88fb32035a84.gradio.live
